# CVAE CPU preprocess

CPU 런타임에서 `filled_samples`를 전수검수하고 CVAE 학습용 manifest/stats를 Drive에 저장한다. GPU 학습 노트북은 이 산출물만 읽는다.


In [ ]:
# CPU runtime에서 먼저 실행
%pip install -q numpy pandas tqdm

from __future__ import annotations
from pathlib import Path
import json
import os
import shutil
import subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


In [ ]:
# Drive setup + local SSD staging
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def find_filled_samples_root(drive_root):
    candidates = [
        drive_root / 'filled_samples',
        drive_root / 'attempt4' / 'cloud_fill_residual_v1' / 'filled_samples',
        drive_root / 'cloud_fill_residual_v1' / 'filled_samples',
    ]
    for cand in candidates:
        count = len(list(cand.glob('*/*.npz'))) if cand.exists() else 0
        print('filled_samples candidate:', cand, 'count=', count)
        if count > 0:
            return cand.resolve(), count
    raise FileNotFoundError(
        'filled_samples not found. Expected /content/drive/MyDrive/filled_samples/{train,val,test}/*.npz '
        'or /content/drive/MyDrive/attempt4/cloud_fill_residual_v1/filled_samples/{train,val,test}/*.npz'
    )

def copy_filled_samples_to_local(src, expected_count, dst=Path('/content/filled_samples')):
    dst = Path(dst)
    existing = len(list(dst.glob('*/*.npz'))) if dst.exists() else 0
    if existing == expected_count:
        print('local filled_samples already staged:', dst, 'count=', existing)
        return dst.resolve()
    if dst.exists():
        shutil.rmtree(dst)
    dst.mkdir(parents=True, exist_ok=True)
    print('copy Drive filled_samples -> local SSD')
    print('src:', src)
    print('dst:', dst)
    # rsync가 있으면 진행률/재시도에 유리하고, 없으면 shutil로 fallback.
    result = subprocess.run(['bash', '-lc', f'rsync -a --info=progress2 "{src}/" "{dst}/"'], text=True)
    if result.returncode != 0:
        print('rsync failed; fallback to shutil.copytree')
        shutil.rmtree(dst, ignore_errors=True)
        shutil.copytree(src, dst)
    copied = len(list(dst.glob('*/*.npz')))
    print('local copied count:', copied)
    if copied != expected_count:
        raise RuntimeError(f'local copy count mismatch: copied={copied}, expected={expected_count}')
    return dst.resolve()

if IN_COLAB:
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ.get('DRIVE_ROOT', '/content/drive/MyDrive'))

DRIVE_SAMPLE_DIR, sample_count = find_filled_samples_root(DRIVE_ROOT)
FILL_RUN_DIR = DRIVE_SAMPLE_DIR.parent
PREPROCESS_DIR = FILL_RUN_DIR / 'cvae_preprocess_v4'
PREPROCESS_DIR.mkdir(parents=True, exist_ok=True)
# 이후 validate/stats는 Drive가 아니라 local SSD에서 읽는다.
SAMPLE_DIR = copy_filled_samples_to_local(DRIVE_SAMPLE_DIR, sample_count)
print('DRIVE_SAMPLE_DIR:', DRIVE_SAMPLE_DIR)
print('LOCAL SAMPLE_DIR:', SAMPLE_DIR)
print('PREPROCESS_DIR:', PREPROCESS_DIR)
print('filled sample count:', sample_count)


In [ ]:
TARGET_CHANNELS = ['lst_k_filled']
EXPECTED_META = ['day_sin', 'day_cos', 'sun_azimuth_norm', 'sun_elevation_norm', 'lat_norm', 'lon_norm']
BANNED_TOKENS = ['air', 'pm10', 'pm25', 'o3', 'pressure', 'press', 'pa', 'ndvi', 'nearest_lst']

def split_from_path(path):
    return Path(path).parent.name

def as_2d_mask(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3:
        return arr[0].astype(bool)
    return arr.astype(bool)

def select_target_and_mask(z, target_indices):
    target_all = z['target'].astype(np.float32)
    mask_all = z['target_mask'].astype(bool)
    return target_all[target_indices], mask_all[target_indices]

def select_target_weight(z, target_indices, target_mask):
    if 'target_weight' in z.files:
        weight = z['target_weight'].astype(np.float32)[target_indices]
        return np.where(target_mask, weight, 0.0).astype(np.float32)
    if 'cloud_fill_uncertainty' in z.files:
        unc = z['cloud_fill_uncertainty'].astype(np.float32)
        if unc.ndim == 2:
            unc = unc[None]
        weight = 1.0 / (1.0 + unc[:1] / 5.0)
        clear = as_2d_mask(z['clear_mask'])
        weight[:, clear] = 1.0
        return np.where(target_mask, weight, 0.0).astype(np.float32)
    return target_mask.astype(np.float32)

def read_channels(path):
    with np.load(path, allow_pickle=True) as z:
        return [str(x) for x in z['condition_channels']], [str(x) for x in z['target_channels']], [str(x) for x in z['meta_channels']]

all_paths = sorted(SAMPLE_DIR.glob('*/*.npz'))
train_paths = [p for p in all_paths if split_from_path(p) == 'train']
val_paths = [p for p in all_paths if split_from_path(p) == 'val']
test_paths = [p for p in all_paths if split_from_path(p) == 'test']
all_paths = train_paths + val_paths + test_paths
print('samples train/val/test:', len(train_paths), len(val_paths), len(test_paths), 'total:', len(all_paths))
if not train_paths:
    raise FileNotFoundError('train filled samples are empty')

CONDITION_CHANNELS, ALL_TARGET_CHANNELS, META_CHANNELS = read_channels(train_paths[0])
TARGET_INDICES = [ALL_TARGET_CHANNELS.index(x) for x in TARGET_CHANNELS]
if META_CHANNELS != EXPECTED_META:
    raise RuntimeError(f'meta channel mismatch: {META_CHANNELS}')
bad = [c for c in CONDITION_CHANNELS + ALL_TARGET_CHANNELS if any(t in c.lower() for t in BANNED_TOKENS)]
if bad:
    raise RuntimeError(f'banned channels found: {bad}')
print('condition channels:', len(CONDITION_CHANNELS), CONDITION_CHANNELS)
print('target channels:', ALL_TARGET_CHANNELS, 'selected:', TARGET_CHANNELS)
print('meta channels:', META_CHANNELS)


In [ ]:
def validate_samples(paths):
    rows = []
    for p in tqdm(paths, desc='validate filled samples'):
        with np.load(p, allow_pickle=True) as z:
            required = ['condition', 'condition_channels', 'target', 'target_channels', 'target_mask', 'meta', 'meta_channels', 'clear_mask', 'cloud_fill_mask']
            missing_keys = [k for k in required if k not in z.files]
            if missing_keys:
                raise RuntimeError(f'missing keys {missing_keys}: {p}')
            cch = [str(x) for x in z['condition_channels']]
            tch = [str(x) for x in z['target_channels']]
            mch = [str(x) for x in z['meta_channels']]
            target, mask = select_target_and_mask(z, TARGET_INDICES)
            weight = select_target_weight(z, TARGET_INDICES, mask)
            clear = as_2d_mask(z['clear_mask'])
            cloud = as_2d_mask(z['cloud_fill_mask'])
            condition = z['condition'].astype(np.float32)
            meta = z['meta'].astype(np.float32)
        if cch != CONDITION_CHANNELS or tch != ALL_TARGET_CHANNELS or mch != META_CHANNELS:
            raise RuntimeError(f'channel mismatch: {p.name}')
        if condition.shape[0] != len(CONDITION_CHANNELS):
            raise RuntimeError(f'condition shape mismatch: {p.name} {condition.shape}')
        if target.shape[0] != len(TARGET_CHANNELS):
            raise RuntimeError(f'target shape mismatch: {p.name} {target.shape}')
        if clear.shape != target.shape[-2:] or cloud.shape != target.shape[-2:]:
            raise RuntimeError(f'mask spatial shape mismatch: {p.name}')
        if not np.isfinite(meta).all():
            raise RuntimeError(f'nonfinite meta: {p.name}')
        if not np.isfinite(target[mask]).all():
            raise RuntimeError(f'nonfinite target in mask: {p.name}')
        nonfinite_by_ch = ~np.isfinite(condition).reshape(condition.shape[0], -1)
        bad_ch_idx = np.where(nonfinite_by_ch.any(axis=1))[0]
        bad_ch_names = [CONDITION_CHANNELS[i] for i in bad_ch_idx]
        rows.append({
            'file': p.relative_to(SAMPLE_DIR).as_posix(),
            'split': p.parent.name,
            'valid_fraction': float(mask.mean()),
            'clear_fraction': float(clear.mean()),
            'cloud_fraction': float(cloud.mean()),
            'mean_weight': float(weight[mask].mean()) if mask.any() else 0.0,
            'condition_nonfinite_pixels': int(nonfinite_by_ch.sum()),
            'condition_nonfinite_channels': '|'.join(bad_ch_names),
        })
    return pd.DataFrame(rows)

validation_df = validate_samples(all_paths)
validation_path = PREPROCESS_DIR / 'dataset_validation.csv'
manifest_path = PREPROCESS_DIR / 'dataset_manifest.csv'
validation_df.to_csv(validation_path, index=False, encoding='utf-8-sig')
validation_df[['file', 'split']].to_csv(manifest_path, index=False, encoding='utf-8-sig')
print('saved:', validation_path)
print('saved:', manifest_path)
print(validation_df.groupby('split').size())
display(validation_df[['valid_fraction', 'clear_fraction', 'cloud_fraction', 'mean_weight']].describe())


In [ ]:
def update_bounds(minima, maxima, values, mask=None):
    values = values.astype(np.float32)
    if mask is not None:
        values = np.where(mask, values, np.nan)
    finite = np.isfinite(values)
    for ch in range(values.shape[0]):
        vals = values[ch][finite[ch]]
        if vals.size:
            minima[ch] = min(minima[ch], float(np.nanpercentile(vals, 0.5)))
            maxima[ch] = max(maxima[ch], float(np.nanpercentile(vals, 99.5)))

def compute_stats(paths):
    cmin = np.full(len(CONDITION_CHANNELS), np.inf)
    cmax = np.full(len(CONDITION_CHANNELS), -np.inf)
    tmin = np.full(len(TARGET_CHANNELS), np.inf)
    tmax = np.full(len(TARGET_CHANNELS), -np.inf)
    mmin = np.full(len(META_CHANNELS), np.inf)
    mmax = np.full(len(META_CHANNELS), -np.inf)
    for p in tqdm(paths, desc='stats from train'):
        with np.load(p, allow_pickle=True) as z:
            target, mask = select_target_and_mask(z, TARGET_INDICES)
            update_bounds(cmin, cmax, z['condition'].astype(np.float32))
            update_bounds(tmin, tmax, target, mask)
            meta = z['meta'].astype(np.float32)
            mmin = np.minimum(mmin, meta)
            mmax = np.maximum(mmax, meta)
    return {
        'condition_channels': CONDITION_CHANNELS,
        'target_channels': TARGET_CHANNELS,
        'source_target_channels': ALL_TARGET_CHANNELS,
        'target_indices': TARGET_INDICES,
        'meta_channels': META_CHANNELS,
        'condition': {n: {'min': float(cmin[i]), 'max': float(cmax[i])} for i, n in enumerate(CONDITION_CHANNELS)},
        'target': {n: {'min': float(tmin[i]), 'max': float(tmax[i])} for i, n in enumerate(TARGET_CHANNELS)},
        'meta': {n: {'min': float(mmin[i]), 'max': float(mmax[i])} for i, n in enumerate(META_CHANNELS)},
    }

stats = compute_stats(train_paths)
for group, names in [('condition', CONDITION_CHANNELS), ('target', TARGET_CHANNELS), ('meta', META_CHANNELS)]:
    bad_bounds = [name for name in names if not np.isfinite(stats[group][name]['min']) or not np.isfinite(stats[group][name]['max'])]
    if bad_bounds:
        raise RuntimeError(f'nonfinite normalization bounds for {group}: {bad_bounds}')
nonfinite_rows = validation_df[validation_df['condition_nonfinite_pixels'] > 0]
if len(nonfinite_rows):
    nonfinite_report = PREPROCESS_DIR / 'condition_nonfinite_report.csv'
    nonfinite_rows.to_csv(nonfinite_report, index=False, encoding='utf-8-sig')
    print('condition nonfinite samples:', len(nonfinite_rows), 'report:', nonfinite_report)
    display(nonfinite_rows.head(20))
stats_path = PREPROCESS_DIR / 'normalization_stats_cvae_v4.json'
stats_path.write_text(json.dumps(stats, ensure_ascii=False, indent=2), encoding='utf-8')
summary = {
    'sample_dir': str(SAMPLE_DIR),
    'preprocess_dir': str(PREPROCESS_DIR),
    'counts': {
        'train': int((validation_df['split'] == 'train').sum()),
        'val': int((validation_df['split'] == 'val').sum()),
        'test': int((validation_df['split'] == 'test').sum()),
        'total': int(len(validation_df)),
    },
    'condition_channels': CONDITION_CHANNELS,
    'target_channels': TARGET_CHANNELS,
    'source_target_channels': ALL_TARGET_CHANNELS,
    'meta_channels': META_CHANNELS,
}
summary_path = PREPROCESS_DIR / 'cvae_preprocess_summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', stats_path)
print('saved:', summary_path)
print('CPU preprocess done. Now switch to GPU runtime and run 02_train_cvae_condprior_residual.ipynb')
